## RNN

In [4]:
import contractions
import pandas as pd
import torch
import re 
from sklearn.model_selection import train_test_split
from torch import nn,optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import TensorDataset,DataLoader
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

def clean_text(text):
    text=str(text).lower()
    text=re.sub(r"<.*?>","",text)
    text=re.sub(r"\n"," ",text)
    text=re.sub(r"\d+","",text)
    text=re.sub(r"https?://\S+|www\.\S+","",text)
    text=re.sub(r"\S+@\S+","",text)
    text=re.sub(r"@\w+","",text)
    text=re.sub(r"#\w+","",text)
    text=re.sub(r"\s+"," ",text)
    text=re.sub(r"[^a-zA-Z0-9\s]","",text)
    return text

train_df=pd.read_csv(r"/Users/suriya/Ukesh_AIML_Projects/Deep_Learning_Comment_Toxicity/Data/train.csv")
train_df['comment_text']=train_df['comment_text'].apply(contractions.fix)
train_df['cleaned_text']=train_df['comment_text'].apply(clean_text)

vocab=set()

for text in train_df['cleaned_text']:
    for word in text.split():
        vocab.add(word)


word2idx={word:idx+1 for idx,word in enumerate(vocab)}

word2idx['<PAD>']=0
word2idx['<UNK>']=len(word2idx)

encode=[]

for sentence in train_df['cleaned_text']:
    temp=[]
    for word in sentence.split():
        if word in word2idx:
            temp.append(word2idx[word])
        else:
            temp.append(word2idx["<UNK>"])
    encode.append(temp)

pad_result=[]

max_len=200

for seq in encode:
    if len(seq)>max_len:
        seq= seq[:max_len]
    else:
        seq=seq+[0]*(max_len-len(seq))
    pad_result.append(seq)


labels=["toxic","severe_toxic","obscene","threat","insult","identity_hate"]

X=torch.tensor(pad_result,dtype=torch.long)
y=torch.tensor(train_df[labels].values,dtype=torch.float32)

x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42)

train_dataset=TensorDataset(x_train,y_train)
test_dataset=TensorDataset(x_test,y_test)

train_loader=DataLoader(train_dataset,batch_size=64,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=64)

class TextRNN(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_size,num_classes):
        super(TextRNN,self).__init__()
        self.embeddings=nn.Embedding(vocab_size,embedding_dim,padding_idx=0)
        self.rnn=nn.RNN(embedding_dim,hidden_size,batch_first=True)
        self.fc=nn.Linear(hidden_size,num_classes)
    
    def forward(self,X):
        X=self.embeddings(X)
        output,hidden=self.rnn(X)
        hidden=hidden.squeeze(0)
        output=self.fc(hidden)
        return output

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


vocab_size=len(word2idx)
embedding_dim=128
hidden_size=64
num_classes=6

model=TextRNN(vocab_size,embedding_dim,hidden_size,num_classes)
criterion=nn.BCEWithLogitsLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)


num_epoch=10

for epoch in range(1,num_epoch+1):
    model.train()
    total_loss=0
    for x_batch,y_batch in train_loader:
        
        x_batch=x_batch.to(device)
        y_batch=y_batch.to(device)
        
        optimizer.zero_grad()
        pred=model(x_batch)
        loss=criterion(pred,y_batch)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    print(f"{epoch}/{num_epoch}--->Loss:---->{total_loss/len(train_loader):.4f}")


model.eval()


with torch.no_grad():
    train_output = torch.sigmoid(model(x_train.to(device)))
    test_output = torch.sigmoid(model(x_test.to(device)))

    train_pred = (train_output > 0.5).int().cpu()
    test_pred = (test_output > 0.5).int().cpu()

    print()

    print(f"Train Accuracy : {accuracy_score(y_train, train_pred)}")
    print(f"Train Precision: {precision_score(y_train, train_pred, average='micro')}")
    print(f"Train Recall   : {recall_score(y_train, train_pred, average='micro')}")
    print(f"Train F1 Score : {f1_score(y_train, train_pred, average='micro')}")

    print()

    print(f"Test Accuracy : {accuracy_score(y_test, test_pred)}")
    print(f"Test Precision: {precision_score(y_test, test_pred, average='micro')}")
    print(f"Test Recall   : {recall_score(y_test, test_pred, average='micro')}")
    print(f"Test F1 Score : {f1_score(y_test, test_pred, average='micro')}")

1/10--->Loss:---->0.1474
2/10--->Loss:---->0.1417
3/10--->Loss:---->0.1396
4/10--->Loss:---->0.1387
5/10--->Loss:---->0.1381
6/10--->Loss:---->0.1223
7/10--->Loss:---->0.1212
8/10--->Loss:---->0.1215
9/10--->Loss:---->0.1149
10/10--->Loss:---->0.1157

Train Accuracy : 0.9003807106598984
Train Precision: 0.906610703043022
Train Recall   : 0.030830716528689694
Train F1 Score : 0.05963350243296407

Test Accuracy : 0.8961616794610685
Test Precision: 0.4156378600823045
Test Recall   : 0.014277636415040994
Test F1 Score : 0.02760694273609403


## LSTM

In [9]:
import contractions
import pandas as pd
import torch
import re 
from sklearn.model_selection import train_test_split
from torch import nn,optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import TensorDataset,DataLoader
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

def clean_text(text):
    text=str(text).lower()
    text=re.sub(r"<.*?>","",text)
    text=re.sub(r"\n"," ",text)
    text=re.sub(r"\d+","",text)
    text=re.sub(r"https?://\S+|www\.\S+","",text)
    text=re.sub(r"\S+@\S+","",text)
    text=re.sub(r"@\w+","",text)
    text=re.sub(r"#\w+","",text)
    text=re.sub(r"\s+"," ",text)
    text=re.sub(r"[^a-zA-Z0-9\s]","",text)
    return text

train_df=pd.read_csv(r"/Users/suriya/Ukesh_AIML_Projects/Deep_Learning_Comment_Toxicity/Data/train.csv")
train_df['comment_text']=train_df['comment_text'].apply(contractions.fix)
train_df['cleaned_text']=train_df['comment_text'].apply(clean_text)

vocab=set()

for text in train_df['cleaned_text']:
    for word in text.split():
        vocab.add(word)


word2idx={word:idx+1 for idx,word in enumerate(vocab)}

word2idx['<PAD>']=0
word2idx['<UNK>']=len(word2idx)

encode=[]

for sentence in train_df['cleaned_text']:
    temp=[]
    for word in sentence.split():
        if word in word2idx:
            temp.append(word2idx[word])
        else:
            temp.append(word2idx["<UNK>"])
    encode.append(temp)

pad_result=[]

max_len=200

for seq in encode:
    if len(seq)>max_len:
        seq= seq[:max_len]
    else:
        seq=seq+[0]*(max_len-len(seq))
    pad_result.append(seq)


labels=["toxic","severe_toxic","obscene","threat","insult","identity_hate"]

X=torch.tensor(pad_result,dtype=torch.long)
y=torch.tensor(train_df[labels].values,dtype=torch.float32)

x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42)

train_dataset=TensorDataset(x_train,y_train)
test_dataset=TensorDataset(x_test,y_test)

train_loader=DataLoader(train_dataset,batch_size=64,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=64)

class TextLSTM(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_size,num_classes):
        super(TextLSTM,self).__init__()
        self.embeddings=nn.Embedding(vocab_size,embedding_dim,padding_idx=0)
        self.lstm=nn.LSTM(embedding_dim,hidden_size,batch_first=True)
        self.fc=nn.Linear(hidden_size,num_classes)
    
    def forward(self,X):
        X=self.embeddings(X)
        output,(hidden,cell_state)=self.lstm(X)
        hidden=hidden.squeeze(0)
        output=self.fc(hidden)
        return output

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


vocab_size=len(word2idx)
embedding_dim=128
hidden_size=64
num_classes=6

model=TextLSTM(vocab_size,embedding_dim,hidden_size,num_classes)
criterion=nn.BCEWithLogitsLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)


num_epoch=10

for epoch in range(1,num_epoch+1):
    model.train()
    total_loss=0
    for x_batch,y_batch in train_loader:
        
        x_batch=x_batch.to(device)
        y_batch=y_batch.to(device)
        
        optimizer.zero_grad()
        pred=model(x_batch)
        loss=criterion(pred,y_batch)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    print(f"{epoch}/{num_epoch}--->Loss:---->{total_loss/len(train_loader):.4f}")


model.eval()


with torch.no_grad():
    train_output = torch.sigmoid(model(x_train.to(device)))
    test_output = torch.sigmoid(model(x_test.to(device)))

    train_pred = (train_output > 0.5).int().cpu()
    test_pred = (test_output > 0.5).int().cpu()

    print()

    print(f"Train Accuracy : {accuracy_score(y_train, train_pred)}")
    print(f"Train Precision: {precision_score(y_train, train_pred, average='micro')}")
    print(f"Train Recall   : {recall_score(y_train, train_pred, average='micro')}")
    print(f"Train F1 Score : {f1_score(y_train, train_pred, average='micro')}")

    print()

    print(f"Test Accuracy : {accuracy_score(y_test, test_pred)}")
    print(f"Test Precision: {precision_score(y_test, test_pred, average='micro')}")
    print(f"Test Recall   : {recall_score(y_test, test_pred, average='micro')}")
    print(f"Test F1 Score : {f1_score(y_test, test_pred, average='micro')}")

1/10--->Loss:---->0.1374
2/10--->Loss:---->0.0884
3/10--->Loss:---->0.0631
4/10--->Loss:---->0.0519
5/10--->Loss:---->0.0434
6/10--->Loss:---->0.0380
7/10--->Loss:---->0.0340
8/10--->Loss:---->0.0310
9/10--->Loss:---->0.0289
10/10--->Loss:---->0.0272

Train Accuracy : 0.9556542583192329
Train Precision: 0.8678431372549019
Train Recall   : 0.868648301455895
Train F1 Score : 0.8682455326889468

Test Accuracy : 0.8992323358922137
Test Precision: 0.7072320650595411
Test Recall   : 0.6884365281311846
Test F1 Score : 0.6977077363896849


## GRU

In [ ]:
import contractions
import pandas as pd
import torch
import re 
from sklearn.model_selection import train_test_split
from torch import nn,optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import TensorDataset,DataLoader
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

def clean_text(text):
    text=str(text).lower()
    text=re.sub(r"<.*?>","",text)
    text=re.sub(r"\n"," ",text)
    text=re.sub(r"\d+","",text)
    text=re.sub(r"https?://\S+|www\.\S+","",text)
    text=re.sub(r"\S+@\S+","",text)
    text=re.sub(r"@\w+","",text)
    text=re.sub(r"#\w+","",text)
    text=re.sub(r"\s+"," ",text)
    text=re.sub(r"[^a-zA-Z0-9\s]","",text)
    return text

train_df=pd.read_csv(r"/Users/suriya/Ukesh_AIML_Projects/Deep_Learning_Comment_Toxicity/Data/train.csv")
train_df['comment_text']=train_df['comment_text'].apply(contractions.fix)
train_df['cleaned_text']=train_df['comment_text'].apply(clean_text)

vocab=set()

for text in train_df['cleaned_text']:
    for word in text.split():
        vocab.add(word)


word2idx={word:idx+1 for idx,word in enumerate(vocab)}

word2idx['<PAD>']=0
word2idx['<UNK>']=len(word2idx)

encode=[]

for sentence in train_df['cleaned_text']:
    temp=[]
    for word in sentence.split():
        if word in word2idx:
            temp.append(word2idx[word])
        else:
            temp.append(word2idx["<UNK>"])
    encode.append(temp)

pad_result=[]

max_len=200

for seq in encode:
    if len(seq)>max_len:
        seq= seq[:max_len]
    else:
        seq=seq+[0]*(max_len-len(seq))
    pad_result.append(seq)


labels=["toxic","severe_toxic","obscene","threat","insult","identity_hate"]

X=torch.tensor(pad_result,dtype=torch.long)
y=torch.tensor(train_df[labels].values,dtype=torch.float32)

x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42)

train_dataset=TensorDataset(x_train,y_train)
test_dataset=TensorDataset(x_test,y_test)

train_loader=DataLoader(train_dataset,batch_size=64,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=64)

class TextGRU(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_size,num_classes):
        super(TextGRU,self).__init__()
        self.embeddings=nn.Embedding(vocab_size,embedding_dim,padding_idx=0)
        self.gru=nn.GRU(embedding_dim,hidden_size,batch_first=True)
        self.fc=nn.Linear(hidden_size,num_classes)
    
    def forward(self,X):
        X=self.embeddings(X)
        output,hidden=self.gru(X)
        hidden=hidden.squeeze(0)
        output=self.fc(hidden)
        return output

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


vocab_size=len(word2idx)
embedding_dim=128
hidden_size=64
num_classes=6

model=TextGRU(vocab_size,embedding_dim,hidden_size,num_classes)
criterion=nn.BCEWithLogitsLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)


num_epoch=10

for epoch in range(1,num_epoch+1):
    model.train()
    total_loss=0
    for x_batch,y_batch in train_loader:
        
        x_batch=x_batch.to(device)
        y_batch=y_batch.to(device)
        
        optimizer.zero_grad()
        pred=model(x_batch)
        loss=criterion(pred,y_batch)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    print(f"{epoch}/{num_epoch}--->Loss:---->{total_loss/len(train_loader):.4f}")


model.eval()


with torch.no_grad():
    train_output = torch.sigmoid(model(x_train.to(device)))
    test_output = torch.sigmoid(model(x_test.to(device)))

    train_pred = (train_output > 0.5).int().cpu()
    test_pred = (test_output > 0.5).int().cpu()

    print()

    print(f"Train Accuracy : {accuracy_score(y_train, train_pred)}")
    print(f"Train Precision: {precision_score(y_train, train_pred, average='micro')}")
    print(f"Train Recall   : {recall_score(y_train, train_pred, average='micro')}")
    print(f"Train F1 Score : {f1_score(y_train, train_pred, average='micro')}")

    print()

    print(f"Test Accuracy : {accuracy_score(y_test, test_pred)}")
    print(f"Test Precision: {precision_score(y_test, test_pred, average='micro')}")
    print(f"Test Recall   : {recall_score(y_test, test_pred, average='micro')}")
    print(f"Test F1 Score : {f1_score(y_test, test_pred, average='micro')}")

1/10--->Loss:---->0.0963
2/10--->Loss:---->0.0497
3/10--->Loss:---->0.0403
4/10--->Loss:---->0.0334
5/10--->Loss:---->0.0280
6/10--->Loss:---->0.0239
7/10--->Loss:---->0.0203
8/10--->Loss:---->0.0174
9/10--->Loss:---->0.0147
10/10--->Loss:---->0.0123

Train Accuracy : 0.9840665538635082
Train Precision: 0.9491258741258741
Train Recall   : 0.9686340279760206
Train F1 Score : 0.9587807290194971

Test Accuracy : 0.9039323202255992
Test Precision: 0.7205925065349985
Test Recall   : 0.7014418999151824
Test F1 Score : 0.7108882521489971


In [24]:
model.eval()

sample="I Love you!!!"
# -----------------------
# Clean the text
# -----------------------

sample = clean_text(sample)

# -----------------------
# Encode
# -----------------------

encoded = []

for word in sample.split():
    encoded.append(word2idx.get(word, word2idx["<UNK>"]))

# -----------------------
# Padding
# -----------------------

MAX_LEN = 200        # Same MAX_LEN used during training

if len(encoded) > MAX_LEN:
    encoded = encoded[:MAX_LEN]
else:
    encoded = encoded + [0] * (MAX_LEN - len(encoded))

# -----------------------
# Convert to Tensor
# -----------------------

sample_tensor = torch.tensor([encoded], dtype=torch.long).to(device)

# -----------------------
# Prediction
# -----------------------

with torch.no_grad():

    output = model(sample_tensor)

    probs = torch.sigmoid(output)

    preds = (probs > 0.5).int()

print("Probabilities:")
print(probs.cpu().numpy())

print()

labels = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

print("Prediction:")

for label, pred, prob in zip(labels, preds[0], probs[0]):
    print(f"{label:15} : {'Yes' if pred else 'No'} ({prob.item():.4f})")

Probabilities:
[[2.6340892e-03 1.8237294e-07 8.2243139e-05 6.6214474e-05 5.1590629e-05
  5.6965899e-05]]

Prediction:
toxic           : No (0.0026)
severe_toxic    : No (0.0000)
obscene         : No (0.0001)
threat          : No (0.0001)
insult          : No (0.0001)
identity_hate   : No (0.0001)


In [ ]:
import pickle

torch.save(model.state_dict(), "tox_model.pth")

with open("word2idx.pkl", "wb") as f:
    pickle.dump(word2idx, f)